# 使用 Admin API 管理你的组织

团队接入 Claude API 时，每次都要重复同样的 Console 操作：邀请每个人、创建工作区、添加成员、查看存在哪些 API 密钥，并为 CI 建立身份。手工完成这些操作不会留下记录，也无法扩展到少数团队之外。

Admin API 把这些控制项开放为 `/v1/organizations` 下的 REST 端点，Python SDK 将其封装为 `client.beta.organization`。本笔记本端到端地完成团队接入，过程中审计组织的 API 密钥，然后删除它创建的全部内容。

**读完本笔记本，你将能够：**

- 以组织管理员身份认证，使用 OAuth 登录或 Console 的 Admin API 密钥
- 邀请用户、设置组织角色，并创建工作区及其成员
- 审计组织的 API 密钥并读取其速率限制
- 为自动化创建服务账号并授予它工作区访问权限

同样的调用在开通脚本或定时审计任务中可原样使用。

## 前提条件

- Python 基础，并熟悉 Claude Console 的组织、工作区和 API 密钥
- Python 3.11+ 和 `anthropic>=1.1.0`，即首个提供 `client.beta.organization` 的版本
- 一个 Claude Console **组织**，你在其中担任 `admin`、`owner` 或 `primary_owner` 角色。Admin API 不适用于个人账户。
- 可选：用于 OAuth 登录路径的 [`ant` CLI](https://platform.claude.com/docs/en/cli-sdks-libraries/cli/authentication)

## 以管理员身份认证

限定于单个工作区的 API 密钥无法调用 Admin API。它接受三种凭证：`org:admin` OAuth 令牌、Admin API 密钥，以及未限定于单个工作区的个人或服务账号密钥。本笔记本使用前两种，而只有组织管理员才能创建二者：

| 凭证 | 获取方式 | 覆盖范围 |
|---|---|---|
| 带 `org:admin` 范围的 OAuth 令牌 | `ant auth login --profile admin --scope org:admin` | 本笔记本中的每个端点 |
| Admin API 密钥（`sk-ant-admin01-...`） | Claude Console → 设置 → [管理员密钥](https://platform.claude.com/settings/admin-keys) | 除服务账号和工作负载身份联合之外的全部内容 |

OAuth 登录会打开浏览器，要求你选择组织，并在 `~/.config/anthropic/` 下存储一个可刷新的令牌。当你使用 `profile="admin"` 构造客户端时，SDK 会读取该令牌。

若改用 Admin API 密钥，在本笔记本旁的 `.env` 文件中添加 `ANTHROPIC_ADMIN_API_KEY=sk-ant-admin01-...`。设置单元格在该密钥已设置时使用它，否则回退到 OAuth 配置。

In [1]:
%%capture
%pip install -U "anthropic>=1.1.0" python-dotenv

In [2]:
import os
import time
from datetime import UTC, datetime
from itertools import islice

import anthropic
from dotenv import load_dotenv

load_dotenv()

ADMIN_KEY = os.getenv("ANTHROPIC_ADMIN_API_KEY")
# The profile name you passed to `ant auth login --profile`.
PROFILE = os.getenv("ANTHROPIC_PROFILE", "admin")
# Only used to pick which rate limit group to print.
MODEL = "claude-sonnet-5"

if ADMIN_KEY:
    client = anthropic.Anthropic(api_key=ADMIN_KEY)
else:
    client = anthropic.Anthropic(profile=PROFILE)

org = client.beta.organization.retrieve()
auth = "an Admin API key" if ADMIN_KEY else f"OAuth profile {PROFILE!r}"
print(f"Connected to {org.name} ({org.id}) with {auth}")

Connected to Example Org (1f6e8a52-93c4-4d7b-a1e0-5b2c9d8e4f37) with OAuth profile 'admin'


`organization.retrieve()` 调用 `GET /v1/organizations/me`。继续之前先核对名称：管理员凭证绑定到一个组织，后续每次调用读取或写入的都是该组织。

笔记本其余部分用一支团队的名称来命名它创建的对象，以便你在 Console 中识别。

In [3]:
RUN_ID = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
TEAM = f"research-{RUN_ID}"
print(TEAM)

research-20260826-180659


## 邀请团队成员并设置角色

邀请会向收件人发送一封含注册链接的邮件，并记录其接受时获得的角色。对于 Console 组织，API 接受 `user`、`developer`、`billing` 和 `claude_code_user`。`admin` 角色只能在 Console 中授予。

新成员从 `user` 开始，该角色授予 Console 和 playground 访问权限而不含 API 密钥，待他们需要更多权限时再提升。

邀请会发送真实的电子邮件。有些组织只接受发往自身邮箱域名的邀请，发往其他域名的邀请会返回 `400`。下一个单元格默认使用现有管理员域名上的一个占位地址，因此在任何组织中都能运行。将 `INVITE_EMAIL` 设为某位团队成员的地址即可尝试完整流程。

In [4]:
INVITE_EMAIL = os.getenv("INVITE_EMAIL")
if not INVITE_EMAIL:
    # Invites to other domains return a 400 in organizations that restrict invite domains.
    users = client.beta.organization.users.list(limit=100)
    admin = next((u for u in users if u.role in {"admin", "owner", "primary_owner"}), None)
    if admin is None:
        raise RuntimeError("No admin user to borrow an email domain from. Set INVITE_EMAIL.")
    INVITE_EMAIL = f"new.teammate+{RUN_ID}@{admin.email.split('@', 1)[1]}"

invite = client.beta.organization.invites.create(email=INVITE_EMAIL, role="user")
print(f"{invite.id}  {invite.email}  role={invite.role}  status={invite.status}")
print(f"expires {invite.expires_at:%Y-%m-%d}")

invite_015gWxHNr6h6TdRPZTmuCGnn  new.teammate+20260826-180659@example.com  role=user  status=pending
expires 2026-09-16


邀请在 21 天后过期。`invites.list()` 返回组织中每条邀请及其状态，`invites.delete()` 撤销一条待处理的邀请。

In [5]:
for inv in client.beta.organization.invites.list(limit=100):
    print(f"{inv.email:<45} {inv.role:<16} {inv.status:<9} sent {inv.invited_at:%Y-%m-%d}")

new.teammate+20260826-180659@example.com                 user             pending   sent 2026-08-26


一旦有人接受，他们会出现在 `users.list()` 中。下一个单元格打印前 10 个。按 `email` 过滤可找到某个人，然后用 `users.update()` 更改其角色，或用 `users.remove()` 将其移除。管理员无法通过 API 移除。

在邀请待处理期间，下一个单元格找不到用户并跳过提升，成员单元格随后也会跳过添加。两者在邀请被接受后都会执行。

In [6]:
# Iterating a list call fetches every page, so cap what gets printed.
for user in islice(client.beta.organization.users.list(limit=100), 10):
    print(f"{user.email:<45} {user.role:<16} joined {user.added_at:%Y-%m-%d}")

teammate = next(iter(client.beta.organization.users.list(email=INVITE_EMAIL)), None)
if teammate is None:
    print(f"\n{INVITE_EMAIL} hasn't accepted yet, nothing to promote")
else:
    teammate = client.beta.organization.users.update(teammate.id, role="developer")
    print(f"\n{teammate.email} is now {teammate.role}")

ada@example.com                              admin            joined 2026-06-01



new.teammate+20260826-180659@example.com hasn't accepted yet, nothing to promote


## 为团队创建工作区

工作区在同一个组织内划分 API 密钥、支出上限、速率限制和使用报告。每个团队或每个环境一个工作区是常见布局。`tags` 承载你自己的元数据，例如成本中心，并在每次读取时返回。

In [7]:
workspace = client.beta.organization.workspaces.create(
    name=TEAM,
    tags={"team": "research", "cost-center": "4711"},
)
print(workspace.id, workspace.name, workspace.tags)

wrkspc_01JwQvzr7rXLA5AGx3HKfFUJ research-20260826-180659 {'team': 'research', 'cost-center': '4711'}


`workspaces.update()` 更改名称、显示颜色或标签。标签会与已有内容合并，将某个标签设为 `None` 会移除它。

In [8]:
workspace = client.beta.organization.workspaces.update(
    workspace.id, tags={"env": "sandbox", "cost-center": None}
)
print(workspace.tags)

{'env': 'sandbox', 'team': 'research'}


组织管理员和 billing 用户是每个工作区的隐式成员，不会出现在 `members.list()` 中。其他所有人需要一个显式的工作区角色，例如 `workspace_user`、`workspace_developer` 或 `workspace_admin`。

`members.update()` 更改角色，`members.remove()` 将某人从工作区移除，而不影响其组织成员身份。

In [9]:
if teammate:
    client.beta.organization.workspaces.members.add(
        workspace.id, user_id=teammate.id, workspace_role="workspace_developer"
    )

members = list(client.beta.organization.workspaces.members.list(workspace.id))
print(f"{workspace.name}: {len(members)} explicit member(s)")
for m in members:
    print(f"  {m.user_id} {m.workspace_role}")

research-20260826-180659: 0 explicit member(s)


## 审计 API 密钥

Admin API 可以列出、重命名、停用和归档 API 密钥，但无法创建它们：新密钥只能来自 Console。把它用于定时审计，而不是用于开通。

下一个单元格遍历组织中每个处于活动状态的密钥，并标记出超过 `MAX_KEY_AGE_DAYS` 的密钥以及没有到期时间的密钥。

In [10]:
MAX_KEY_AGE_DAYS = 90
now = datetime.now(UTC)

# The default workspace isn't returned by workspaces.list(), so any ID missing here is it.
workspace_names = {w.id: w.name for w in client.beta.organization.workspaces.list(limit=1000)}

flagged = []
for key in client.beta.organization.api_keys.list(status="active", limit=1000):
    if key.scope.type == "workspace":
        where = workspace_names.get(key.scope.workspace_id, "Default")
    else:
        where = "(organization)"
    age = (now - key.created_at).days
    expires = f"{key.expires_at:%Y-%m-%d}" if key.expires_at else "never"
    flags = []
    if age > MAX_KEY_AGE_DAYS:
        flags.append("old")
    if key.expires_at is None:
        flags.append("no-expiry")
    if flags:
        flagged.append(key)
    hint = key.partial_key_hint or "-"
    print(f"{key.name:<28} {where:<18} {hint:<24}", end=" ")
    print(f"{age:>4}d  expires {expires:<10} {' '.join(flags)}")

print(f"\n{len(flagged)} key(s) flagged")

backend-prod                 production         sk-ant-api03-R2D...igAA    14d  expires 2027-08-12 
ci-deploy                    production         sk-ant-api03-9xQ...0mAA    14d  expires 2026-12-09 

0 key(s) flagged


停用密钥是可逆的（`status="active"` 会重新启用它）。归档则不可逆。试运行时保持 `APPLY = False`，或设为 `True` 以停用被标记的密钥。

先读取被标记的清单。没有到期时间的密钥即使服务于生产流量也会被标记，而被停用的密钥会立即开始导致请求失败。

In [11]:
APPLY = False

if not flagged:
    print("no flagged keys, nothing to do")
for key in flagged:
    if APPLY:
        key = client.beta.organization.api_keys.update(key.id, status="inactive")
    print(f"{key.name:<28} -> {key.status}{'' if APPLY else '  (dry run, unchanged)'}")

no flagged keys, nothing to do


## 为自动化创建服务账号

服务账号是组织中非人类的成员，用于 CI 任务和后端服务。它像人一样持有组织角色和工作区角色，因此签发给它的凭证不依赖某位员工的账号。服务账号端点只接受 `org:admin` OAuth 凭证。Admin API 密钥在这里会被拒绝。

`name` 是由小写字母、数字和连字符组成的短标识，并且必须在组织内唯一。服务账号端点可能滞后于创建。几秒前创建的对象在这些端点可能短暂返回 `404`，或在列表中缺失。当某个调用引用你刚创建的对象时，遇到 `404` 应重试，下一个单元格中的辅助函数正是这样做的。

In [12]:
# New objects can briefly 404 on the service account endpoints.
def retry_on_not_found(call, attempts: int = 6, delay: float = 5.0):
    for attempt in range(attempts):
        try:
            return call()
        except anthropic.NotFoundError:
            if attempt == attempts - 1:
                raise
            time.sleep(delay)


service_account = None
if ADMIN_KEY:
    print("Skipped: service account endpoints need the org:admin OAuth credential")
else:
    service_account = client.beta.organization.service_accounts.create(
        name=f"ci-{TEAM}",
        description="Deploy pipeline for the research team",
    )
    print(service_account.id, service_account.name, service_account.organization_role)

svac_01ABCDEFabcdef0123456789XY ci-research-20260826-180659 developer


为它在团队工作区授予角色，然后列出它的成员身份。每个服务账号同时也是默认工作区中的隐式 `workspace_user`，几秒前授予的角色可能需要一点时间才出现在列表中。

In [13]:
if service_account:
    grant = retry_on_not_found(
        lambda: client.beta.organization.service_accounts.workspaces.add(
            service_account.id, workspace_id=workspace.id, workspace_role="workspace_developer"
        )
    )
    print(f"granted {grant.workspace_role} in {workspace.name}\n")
    for m in client.beta.organization.service_accounts.workspaces.list(service_account.id):
        name = workspace.name if m.workspace_id == workspace.id else "Default"
        print(f"{name:<28} {m.workspace_role:<22} {'implicit' if m.implicit else 'explicit'}")

granted workspace_developer in research-20260826-180659

Default                      workspace_user         implicit
research-20260826-180659     workspace_developer    explicit


要让某条流水线以该服务账号身份认证而无需存储密钥，请将你的 CI 提供商注册为联合签发方，并添加一条把它的 OIDC 令牌映射到该账号的规则。[工作负载身份联合](https://platform.claude.com/docs/en/manage-claude/workload-identity-federation) 说明了该设置。

## 查看速率限制

速率限制按组织设置。每个分组覆盖一个模型系列或 API 的另一部分，例如 `batch`、`files` 或 `web_search`。`rate_limits.list()` 返回每个分组。下一个单元格打印覆盖 `MODEL` 的分组（该值由设置单元格定义），以及与模型无关的分组。

In [14]:
for rl in client.beta.organization.rate_limits.list():
    if rl.group_type == "model_group" and MODEL not in (rl.models or []):
        continue
    label = MODEL if rl.group_type == "model_group" else rl.group_type
    limits = ", ".join(f"{limit.type}={limit.value:,}" for limit in rl.limits)
    print(f"{label:<18} {limits}")

batch              enqueued_batch_requests=500,000, requests_per_minute=4,000
claude-sonnet-5    input_tokens_per_minute=10,000,000, output_tokens_per_minute=2,000,000, requests_per_minute=10,000


工作区继承这些值。管理员可以在 Console 中为工作区设置更低的上限，而 `workspaces.rate_limits.list()` 只返回被覆盖的值。

In [15]:
overrides = list(client.beta.organization.workspaces.rate_limits.list(workspace.id))
if not overrides:
    print(f"{workspace.name} has no overrides and inherits the organization limits")
for rl in overrides:
    print(rl.group_type, rl.models, [(limit.type, limit.value) for limit in rl.limits])

research-20260826-180659 has no overrides and inherits the organization limits


## 清理

删除本笔记本创建的内容。**归档工作区是永久性的，并会撤销其中每个 API 密钥**，因此在针对真实组织运行之前请再次核对 `workspace.name`。已归档的工作区不计入 100 个工作区的上限。

如果前面的单元格抛出异常，请手工运行本单元格。它需要内核中的 `workspace`、`service_account` 和 `invite`。

In [16]:
archived = client.beta.organization.workspaces.archive(workspace.id)
print(f"archived workspace {archived.name} at {archived.archived_at:%Y-%m-%d %H:%M}")

if service_account:
    retry_on_not_found(
        lambda: client.beta.organization.service_accounts.archive(service_account.id)
    )
    print(f"archived service account {service_account.name}")

# Deleting an accepted invite returns a 400, because that person is a member now.
if client.beta.organization.invites.retrieve(invite.id).status == "pending":
    client.beta.organization.invites.delete(invite.id)
    print(f"deleted invite for {invite.email}")
else:
    print(f"{invite.email} joined the organization, offboard them with users.remove()")

archived workspace research-20260826-180659 at 2026-08-26 18:07
archived service account ci-research-20260826-180659
deleted invite for new.teammate+20260826-180659@example.com


## 后续步骤

你以管理员身份完成了认证，邀请了一位用户，创建了工作区，为服务账号授予了访问权限，审计了组织的 API 密钥，并读取了其速率限制，全部通过 `client.beta.organization` 完成。

此处未涉及的其他相关管理界面：

- [用量与成本 cookbook](../observability/usage_cost_api.ipynb) 以及 [用量与成本 API](https://platform.claude.com/docs/en/manage-claude/usage-cost-api)：按工作区、密钥和模型统计词元与金额
- [Claude Code Analytics API](https://platform.claude.com/docs/en/manage-claude/claude-code-analytics-api)：组织内每位用户的 Claude Code 活动
- [工作负载身份联合](https://platform.claude.com/docs/en/manage-claude/workload-identity-federation)：通过 `client.beta.organization.federation` 为你创建的服务账号实现免密钥认证
- [Admin API 参考](https://platform.claude.com/docs/en/api/admin)：每个端点，包括 SDK 在 `external_keys` 和 `federation` 下暴露的那些